# 9장. 회귀 분석으로 숫자 예측하기

이 Notebook의 핵심은 높은 점수를 만드는 것이 아니라 다음 순서를 지키는 것입니다.

`예측 시점 → 누수 방지 → 시간 분할 → Train TimeSeriesSplit → 모델 선택 고정 → Final Test → Baseline 비교`

**Final Test 결과를 보고 모델을 고르지 않습니다.**

## 0. 실행 전 확인

Chapter09는 Chapter05의 오류 탐지용 전용 Raw가 아니라 **공통 프로젝트 `data/raw`**를 사용합니다. 먼저 프로젝트 루트에서 다음 명령을 실행해 관계 검증을 통과한 모델링 입력을 준비합니다.

```powershell
python scripts/prepare_ch09_data.py
```

이 명령은 공통 `data/raw`를 전처리하고 FK 관계를 검증한 뒤 `data/processed/*_clean.csv`를 생성합니다.

In [1]:
from pathlib import Path
import sys
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

from src.regression import (
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    TARGET_COLUMN,
    build_feature_audit,
    build_leakage_checklist,
    build_regression_dataset,
    build_regression_validation,
    build_split_summary,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    make_regression_models,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)

PROJECT_ROOT: C:\dev\llm-data-analysis-study\llm-data-analysis-course
PROCESSED_DIR: C:\dev\llm-data-analysis-study\llm-data-analysis-course\data\processed


## 1. 예측 문제와 예측 시점

- Target: 주문 1건의 주문 상세 금액 합계 `order_total`
- 예측 시점: 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만 주문 상세 수량·단가·금액은 모델에 제공하지 않는 시점
- 목표 계산 재료와 사후 정보, 식별자는 feature에서 제외합니다.

In [2]:
print('Target:', TARGET_COLUMN)
print('Allowed features:', FEATURE_COLUMNS)
print('Forbidden features:', sorted(FORBIDDEN_FEATURES))
validate_feature_columns()
print('Feature leakage gate: PASS')

Target: order_total
Allowed features: ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']
Forbidden features: ['avg_unit_price', 'customer_id', 'item_count', 'line_total', 'order_id', 'order_status', 'order_total', 'product_id', 'quantity', 'total_quantity', 'unit_price']
Feature leakage gate: PASS


## 2. 검증된 전처리 데이터를 불러옵니다

In [3]:
data = load_regression_source_data(PROCESSED_DIR)
for name, frame in data.items():
    print(name, frame.shape)

customers (150, 6)
orders (300, 7)
order_items (764, 6)


## 3. 주문 단위 Target을 만들고 관계를 검증합니다

`line_total = quantity × unit_price`를 확인하고, 주문과 Target은 `one_to_one`, 주문과 고객은 `many_to_one` 관계가 깨지지 않는지 확인합니다.

In [4]:
model_data = build_regression_dataset(
    customers=data['customers'],
    orders=data['orders'],
    order_items=data['order_items'],
)
display(model_data.head())
print('Model rows:', len(model_data))

,order_id,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek,order_total,gender,age,city
0,238,124,2025-09-16,bank_transfer,completed,9,1,361000,M,67,대구
1,79,62,2025-09-18,bank_transfer,refunded,9,3,283000,F,63,광주
2,12,98,2025-09-19,kakao_pay,completed,9,4,444000,F,61,인천
3,169,9,2025-09-19,kakao_pay,completed,9,4,448000,M,69,서울
4,152,62,2025-09-20,naver_pay,refunded,9,5,985000,F,63,광주


Model rows: 300


## 4. Feature Audit을 확인합니다

In [5]:
feature_audit = build_feature_audit()
display(feature_audit)

,column,selected,role,reason
0,order_month,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
1,order_dayofweek,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
2,age,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
3,payment_method,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
4,gender,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
5,city,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
6,avg_unit_price,False,forbidden,주문 상세에서 만든 목표 대리 변수
7,customer_id,False,forbidden,고객 식별자
8,item_count,False,forbidden,주문 상세가 확인된 뒤 계산되는 사후 집계
9,line_total,False,forbidden,목표값을 구성하는 주문 상세 금액


## 5. 시간 순서로 Train / Final Test를 분리합니다

같은 달력 날짜를 양쪽에 나누지 않습니다. Final Test는 모델 선택이 끝날 때까지 사용하지 않습니다.

In [6]:
train_data, test_data = split_model_data_by_time(model_data, test_size=0.2)
split_summary = build_split_summary(train_data, test_data)
display(split_summary)
assert train_data['order_date'].max().normalize() < test_data['order_date'].min().normalize()
print('Chronological split gate: PASS')

,split,rows,ratio_pct,start_date,end_date
0,train,244,81.33,2025-09-16,2026-06-21
1,test,56,18.67,2026-06-22,2026-09-15


Chronological split gate: PASS


In [7]:
X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()
print('Train:', X_train.shape, 'Final Test:', X_test.shape)

Train: (244, 6) Final Test: (56, 6)


## 6. Pipeline 후보 모델을 준비합니다

숫자형 결측 대체·표준화와 범주형 결측 대체·One-Hot Encoding은 모델 Pipeline 안에서 각 Train Fold로만 학습됩니다.

In [8]:
models = make_regression_models(random_state=42)
print(list(models))

['Baseline Mean', 'Linear Regression', 'Random Forest']


## 7. Train 기간의 TimeSeriesSplit으로 후보를 비교합니다

여기까지는 Final Test의 `y_test`를 모델 선택에 사용하지 않습니다.

In [9]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
)
display(cv_summary)

,model,n_splits,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Baseline Mean,5,423004.808743,44153.043466,-0.022821
1,Random Forest,5,430299.941757,20766.556459,-0.096725
2,Linear Regression,5,470993.023937,77230.384346,-0.368772


## 8. 비베이스라인 모델 선택을 고정합니다

Linear Regression과 Random Forest 중 Train CV의 평균 MAE가 더 낮은 모델을 선택합니다. **이 시점까지 Final Test 성능은 보지 않습니다.**

In [10]:
selected_model_name = select_diagnostic_model(cv_summary)
print('Frozen selected model:', selected_model_name)

Frozen selected model: Random Forest


## 9. 이제 Final Test에서 Baseline과 고정 모델만 평가합니다

Final Test 결과가 기대와 다르더라도 같은 Test를 보고 다른 후보로 교체하지 않습니다.

In [11]:
model_comparison, predictions = train_and_evaluate_models(
    models=models,
    selected_model_name=selected_model_name,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)
display(model_comparison)

,model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,416266.191884,443449.648712,522189.69962,-0.001121,0.00
1,Random Forest,selected_by_train_cv,325118.142627,463973.531438,544352.05636,-0.087902,-4.63


### 지표 읽기

- MAE: 평균적으로 얼마만큼 틀리는가
- RMSE: 큰 오차에 더 민감
- R²: 평균 예측 대비 설명력, 음수가 될 수 있음
- MAE 개선율: Dummy baseline 대비 고정 모델의 Final Test MAE 개선 정도

## 10. 실제값·예측값·잔차를 진단합니다

In [12]:
prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)
display(prediction_result.head(10))
print('주의: order_id가 포함된 이 표는 내부 진단용입니다.')

,order_id,order_date,actual_order_total,predicted_order_total,residual,abs_error,model
0,51,2026-08-23,1877000,5.451749e+05,1.331825e+06,1.331825e+06,Random Forest
1,21,2026-08-27,1767000,7.492445e+05,1.017755e+06,1.017755e+06,Random Forest
2,96,2026-07-19,1690000,6.814601e+05,1.008540e+06,1.008540e+06,Random Forest
3,145,2026-08-04,1863000,8.928091e+05,9.701909e+05,9.701909e+05,Random Forest
4,183,2026-07-31,1685000,7.819218e+05,9.030782e+05,9.030782e+05,Random Forest
5,15,2026-07-31,1677000,8.241877e+05,8.528123e+05,8.528123e+05,Random Forest
6,168,2026-08-19,10000,8.495438e+05,-8.395438e+05,8.395438e+05,Random Forest
7,194,2026-07-27,257000,1.075221e+06,-8.182207e+05,8.182207e+05,Random Forest
8,282,2026-07-02,1566000,7.680465e+05,7.979535e+05,7.979535e+05,Random Forest
9,179,2026-09-15,86000,8.644396e+05,-7.784396e+05,7.784396e+05,Random Forest


주의: order_id가 포함된 이 표는 내부 진단용입니다.


## 11. 자동 Validation Evidence를 만듭니다

In [13]:
validation = build_regression_validation(
    train_data=train_data,
    test_data=test_data,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
)
display(validation)
assert validation['status'].eq('PASS').all()

,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 12. Evidence·내부 진단·공개 보고서를 구분해 저장합니다

In [14]:
checklist = build_leakage_checklist()
output_paths = save_regression_outputs(
    model_data=model_data,
    split_summary=split_summary,
    feature_audit=feature_audit,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
    prediction_result=prediction_result,
    validation=validation,
    checklist=checklist,
    report_dir=REPORT_DIR,
)
for name, path in output_paths.items():
    print(name, '->', path)

model_data_internal -> C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch09_regression_model_data_internal.csv
split_summary -> C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch09_regression_split_summary.csv
feature_audit -> C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch09_regression_feature_audit.csv
cv_summary -> C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch09_regression_cv_summary.csv
model_comparison -> C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch09_regression_model_comparison.csv
predictions_internal -> C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch09_regression_predictions_internal.csv
validation -> C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch09_regression_validation.csv
checklist -> C:\dev\llm-data-analysis-study\llm-data-analysis-course\reports\ch09_regression_checklist.csv
report -> C:\dev\llm-data-analysis-study\llm-data-analysis-course\re

## 13. 전체 공통 함수를 새 실행으로 다시 확인합니다

셀의 메모리 상태가 아니라 공통 함수가 같은 순서를 재현하는지 확인합니다.

In [15]:
result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)
print('Selected by Train CV:', result['selected_model_name'])
display(result['validation'])

Selected by Train CV: Random Forest


,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 14. 해석

낮은 성능이나 음수 R²도 숨기지 않습니다. 현재 예측 시점에 허용한 정보가 약하다면 `현재 데이터로는 모델 사용 보류`도 올바른 결론입니다.

LLM이 만든 코드도 같은 예측 시점·누수·Pipeline·Train CV·Final Test 보호 기준으로 검토합니다.

## 15. 완료 체크

- [x] `python scripts/prepare_ch09_data.py`로 공통 모델링 입력을 준비했다.
- [x] Target과 예측 시점을 설명할 수 있다.
- [x] 금지 feature가 입력에 없음을 확인했다.
- [x] 같은 날짜가 Train과 Final Test에 동시에 들어가지 않는다.
- [x] 전처리는 Pipeline 내부에서 학습된다.
- [x] Train TimeSeriesSplit으로 후보를 비교했다.
- [x] Final Test 전에 Selected Model을 고정했다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했다.
- [x] MAE, RMSE, R²를 함께 읽었다.
- [x] 내부 식별자 결과와 공개 보고서를 구분했다.
- [x] Validation Evidence가 모두 PASS다.

## 제출 정보
- 이름: 권혁수
- GitHub ID: kwonhyeoksoo
- 작성일: 2026-11-11
- 최종 Notebook URL: https://github.com/kwonhyeoksoo/llm-data-analysis-study/blob/main/chapter09/chapter09.ipynb

## 1. 예측 문제 정의
- Target: order_total (같은 order_id의 line_total 합계, line_total = quantity × unit_price)
- 예측 단위: 주문 1건
- 예측 시점: 주문이 들어오는 시점. 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만 주문 상세(수량, 단가, 금액)는 아직 알 수 없다고 가정
- 예측 시점에 알 수 있는 정보: order_month, order_dayofweek, age, payment_method, gender, city
- 예측 시점에 알 수 없는 정보: quantity, unit_price, line_total, item_count, total_quantity, avg_unit_price, order_status, 그리고 order_total 자체

### 나의 해석과 판단
금액은 주문 상세가 확정되어야 계산되므로 그 이전 시점의 정보만으로 얼마나 맞힐 수 있는지를 보는 문제로 정했다. 상세를 알고 있으면 합계를 계산하면 되어 예측이 필요 없기 때문에 이 시점이 의미가 있다. 다만 나이, 성별, 도시를 주문 시점에 안다는 것은 교육용 가정이라고 판단했다.

---

## 2. Feature Leakage Audit

| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| --- | --- | --- | --- | --- | --- |
| order_month | O | X | X | 사용 | 주문 시점에 알 수 있음 |
| order_dayofweek | O | X | X | 사용 | 주문 시점에 알 수 있음 |
| age | O(가정) | X | X | 사용 | 고객 정보 |
| payment_method | O | X | X | 사용 | 주문 시점에 선택 |
| gender | O(가정) | X | X | 사용 | 고객 정보 |
| city | O(가정) | X | X | 사용 | 고객 정보 |
| quantity | X | O | X | 제외 | 목표값 계산 재료 |
| unit_price | X | O | X | 제외 | 목표값 계산 재료 |
| line_total | X | O | X | 제외 | 목표값을 구성하는 주문 상세 금액 |
| item_count | X | X | 사후 집계 | 제외 | 주문 상세가 확인된 뒤 계산 |
| total_quantity | X | 대리 변수 | 사후 집계 | 제외 | 주문 상세에서 만든 목표 대리 변수 |
| avg_unit_price | X | 대리 변수 | 사후 집계 | 제외 | 주문 상세에서 만든 목표 대리 변수 |
| order_status | X | X | 사후 정보 | 제외 | 예측 시점 이후에 확정될 수 있음 |
| order_id, customer_id, product_id | X | X | ID | 제외 | 식별자이며 product_id는 주문 상세가 있어야 알 수 있음 |
| order_total | X | Target | X | 제외 | 예측 대상 자체 |

- 금지 feature overlap 개수: 0
- 가장 위험하다고 판단한 feature: line_total (quantity, unit_price 포함)
- 이유: order_total이 이 값들의 합이라 넣으면 정답을 입력하는 것과 같기 때문에 높은 점수가 나와도 실제 예측 능력이 아니다.

![Leakage 점검](images/step02_leakage.png)

---

## 3. Target 생성과 관계 검증

### 계산 기준
```text
line_total = quantity × unit_price
order_total = order_id별 line_total 합계
```
- line_total 검증 결과: 전처리 단계에서 quantity × unit_price와 일치 확인
- orders ↔ target 관계 검증 결과: one_to_one 통과, 모델 데이터 300행으로 주문 수와 같음
- orders → customers 관계 검증 결과: many_to_one 통과, 행 손실 없음

### 나의 해석과 판단
관계 검증 없이 병합하면 키가 중복될 때 행이 늘어 order_total이 부풀려지거나 매칭 실패로 행이 조용히 사라질 수 있다. 모델 행 수 300과 주문 수 300의 일치로 이를 확인했다.

---

## 4. Train / Final Test 분할
- Train 시작일: 2025-09-16
- Train 종료일: 2026-06-21
- Train 행 수: 244
- Final Test 시작일: 2026-06-22
- Final Test 종료일: 2026-09-15
- Final Test 행 수: 56
- 같은 달력 날짜 중복 여부: 없음 (Train 종료일 < Test 시작일, gate PASS)

### 분할 판단
실제 예측은 과거로 학습해 미래를 맞히는 일이다. 랜덤 분할을 쓰면 미래 주문이 학습에 섞여 성능이 실제보다 좋아 보일 수 있어 시간 순서 분할을 사용했다. 한계로 Test가 약 3개월(56행)이라 계절 패턴이 Train과 다를 수 있고 지표가 흔들릴 수 있다.

![시간 분할](images/step04_split.png)

---

## 5. Train TimeSeriesSplit 후보 비교

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고 |
| --- | ---: | ---: | ---: | --- |
| Baseline Mean | 423,005 | 44,153 | -0.023 | 기준 |
| Linear Regression | 470,993 | 77,230 | -0.369 | 평균·변동 모두 가장 나쁨 |
| Random Forest | 430,300 | 20,767 | -0.097 | 비베이스라인 중 평균 MAE 최저, 표준편차 최소 |

### Train CV로 선택한 비베이스라인 모델
- Selected Model: Random Forest
- 선택 기준: 비베이스라인 후보 중 CV MAE 평균이 더 낮은 모델
- Final Test 결과를 보기 전에 선택했는가: 예

### 나의 해석과 판단
평균만 보면 fold별 변동을 놓친다. 평균이 비슷해도 표준편차가 크면 구간에 따라 성능이 들쭉날쭉해 믿기 어렵다. Linear Regression은 표준편차가 77,230으로 가장 불안정했다. 또 Baseline이 CV 평균 MAE에서 가장 낮아 Train 단계에서 이미 복잡한 모델이 Baseline보다 낫다는 증거가 없었다. fold 5개, 244행이라 이 값 자체도 불안정하다.

---

## 6. Final Test: Baseline vs Frozen Model

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Baseline Mean | baseline | 416,266 | 443,450 | 522,190 | -0.001 | 0.00% |
| Frozen Selected Model (Random Forest) | selected_by_train_cv | 325,118 | 463,974 | 544,352 | -0.088 | -4.63% |

![최종 회귀 평가](images/step06_final_metrics.png)

### 결과 관찰
고정한 Random Forest는 Test MAE, RMSE, R² 모두 Baseline보다 나빴다. Train MAE(325,118)는 Baseline보다 훨씬 낮은데 Test MAE는 더 높았다.

### 나의 해석과 판단
- Frozen Model이 baseline보다 실제로 개선되었는가? 아니오. MAE가 Baseline보다 약 4.63% 나쁘다. Train에서만 잘 맞고 Test에서 나빠져 과적합으로 보이며 CV에서 보인 방향이 Final Test에서도 재현되었다.
- MAE와 RMSE 차이: RMSE가 MAE보다 약 17~18% 커서 일부 큰 오차가 있다.
- R²: Baseline이 거의 0이고 Random Forest는 -0.088이라 평균 예측보다 설명력이 낮다. 코드 오류가 아니라 허용 feature가 금액을 설명하지 못한다는 신호이다.
- Final Test 결과를 보고 모델을 바꾸지 않았다.

### 업무·분석적 의미
평균 오차가 약 46만 원인데 주문 금액이 수십만~백만 원대여서 실사용 가치가 없다. 다만 Test가 56행이라 -4.63% 차이는 우연일 수 있고, 정확히는 'Baseline보다 낫다는 증거가 없다'는 의미이다.

---

## 7. 대표 오차 사례

### 사례 1
- 실제값: 1,877,000
- 예측값: 약 545,175
- residual: +1,331,825
- absolute error: 1,331,825
- 관찰: Test에서 가장 큰 오차이며 실제가 예측의 약 3.4배인 큰 과소예측
- 가능한 이유 후보: 주문 금액은 상품 수와 수량에 크게 좌우되는데 그 정보를 feature에서 뺐기 때문에 고액 주문을 구분하지 못했을 수 있다
- 추가 확인: 이 주문의 상품 수·수량과, 같은 결제수단/도시/연령대 주문의 금액 분포 비교

### 사례 2
- 실제값: 10,000
- 예측값: 약 849,544
- residual: -839,544
- absolute error: 839,544
- 관찰: 소액 주문을 85만 원대로 예측한 큰 과대예측
- 가능한 이유 후보: 모델이 55만~110만 원대 중간값으로 수렴해 극단적 소액·고액을 따라가지 못했을 수 있다
- 추가 확인: 소액 주문의 공통 특징(상품 수 1개 등)과 feature와의 관계

> 위 이유는 데이터로 확인하지 않은 후보이며 원인으로 단정하지 않는다.

---

## 8. Validation Evidence

| check | value | status |
| --- | --- | --- |
| forbidden_feature_overlap | 0 | PASS |
| strict_train_before_test | True | PASS |
| selected_model_exists_in_train_cv | True | PASS |
| final_test_contains_baseline | True | PASS |
| final_test_contains_frozen_selected_model | True | PASS |
| test_rows_for_r2 | 56 | PASS |

### FAIL 항목
- 없음. 다만 PASS는 절차를 지켰다는 뜻이며 모델이 좋다는 뜻은 아니다.

---

## 9. Evidence와 공개 범위

### 생성된 Evidence
- [x] `ch09_regression_split_summary.csv`
- [x] `ch09_regression_feature_audit.csv`
- [x] `ch09_regression_cv_summary.csv`
- [x] `ch09_regression_model_comparison.csv`
- [x] `ch09_regression_validation.csv`
- [x] `ch09_regression_checklist.csv`

### Internal 파일
- [x] `ch09_regression_model_data_internal.csv`
- [x] `ch09_regression_predictions_internal.csv`

### 공개 결과
- [x] `ch09_regression_report.md`
- [x] `ch09_actual_vs_predicted.png`
- [x] `ch09_residual_histogram.png`

### 공개 범위 판단
order_id가 포함된 예측 결과는 개별 주문과 금액을 가리킬 수 있어 Internal로 구분했다. 공개 보고서에는 order_id 없이 집계 지표와 그래프만 둔다.

---

## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 데이터로는 사용 보류

### 판단 근거
1. Train CV에서 Baseline이 MAE 평균이 가장 낮았고, Final Test에서도 Random Forest가 Baseline보다 MAE -4.63%, R² -0.088로 나빴다.
2. 누수 점검, 시간 분할, 모델 고정, Validation은 모두 통과했으므로 성능이 낮은 것이 절차 문제가 아니라 허용 feature의 정보 부족으로 보인다.
3. Test 56행에서 상위 오차가 큰 과소/과대예측으로 나타났고 평균 오차(약 46만 원)가 실사용 기준에 비해 크다.

### 업무적 위험
이 모델 예측을 금액 추정에 쓰면 평균보다 나은 근거 없이 큰 오차를 가진 숫자를 신뢰하게 된다.

### 현재 데이터의 한계
주문 단위 300건, 약 12개월 데이터이며 Test는 56행이다. 허용 feature가 가정에 기반하고 주문 상세 정보가 없다.

### 다음 개선 우선순위
1. 예측 시점에 실제로 알 수 있는 정보(장바구니 상품 수, 할인, 이전 구매 이력 등)가 있는지 확인
2. 더 긴 기간과 더 많은 주문 데이터로 Test 크기 확대
3. 금액이 극단인 주문을 별도로 다루는 방법 검토

---

## 11. 전체 재실행 확인

실행 명령 (가이드 명령을 `-m` 형태로 실행):
```powershell
python -m scripts.prepare_ch09_data
python -m scripts.run_regression_analysis
```
- 실행 성공 여부: 성공 ("9장 회귀 분석 완료" 출력)
- Train CV Selected Model: Random Forest
- Final Validation 전체 PASS 여부: 6개 모두 PASS
- Notebook 결과와 스크립트 결과의 일관성: 일치. CV 표, Final Test 지표, Validation, 오차 상위 10건이 같았다. 노트북 섹션 13 재실행도 같은 선택 모델과 PASS였다.

---

## 최종 체크
- [x] Target과 예측 시점을 정의했습니다.
- [x] Leakage Audit을 수행했습니다.
- [x] Target 계산과 관계 검증을 확인했습니다.
- [x] Train과 Final Test가 시간 순서로 엄격히 분리되었습니다.
- [x] 전처리가 Pipeline 내부에서 학습됩니다.
- [x] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [x] Final Test 전에 Selected Model을 고정했습니다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [x] MAE, RMSE, R²를 함께 해석했습니다.
- [x] 대표 오차 사례를 관찰과 가설로 구분했습니다.
- [x] Validation Evidence가 모두 PASS입니다.
- [x] Internal 결과와 공개 결과를 구분했습니다.
- [x] 낮은 성능도 숨기지 않았습니다.
- [x] 전체 스크립트를 재실행했습니다.
- [x] 최종 Notebook URL을 제출합니다.